In [1]:
from pathlib import Path
import torch
import torch.nn as nn
from config import get_config, latest_weights_file_path
from train import get_model, get_ds, run_validation
from translate import translate
from train import greedy_decode

/home/sarthak/miniconda3/envs/torch310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

Using device: cuda


Max length of source sentence: 434
Max length of target sentence: 288


In [3]:
model_filename = latest_weights_file_path(config)
print("Model path:", model_filename)

Model path: iitb_weights/tmodel14.pt


In [4]:
# Load the pretrained weights
model_filename = latest_weights_file_path(config)
state = torch.load(model_filename)
model.load_state_dict(state['model_state_dict'])

<All keys matched successfully>

In [5]:
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: print(msg), 0, None, num_examples=10)

stty: Inappropriate ioctl for device


--------------------------------------------------------------------------------
    SOURCE: Source
    TARGET: स्रोत
 PREDICTED: स्रोत
--------------------------------------------------------------------------------
    SOURCE: grin
    TARGET: मुस्कुराहट
 PREDICTED: 
--------------------------------------------------------------------------------
    SOURCE: It was a complete justification of the work of Asutosh.
    TARGET: यह आशुतोष के द्वारा किया गये कार्यों का पूरा समर्थन था। 
 PREDICTED: यह आशुतोष के द्वारा किया गया था ।
--------------------------------------------------------------------------------
    SOURCE: Up to 20 times of Net Monthly Salary on the basis of Last Salary Drawn, subject to ‘Deduction Norms’
    TARGET: कटौती मानदण्डों की शर्त के अधीन अंतिम आहरित वेतन के आधार पर निवल मासिक वेतन के 20 गुणा तक
 PREDICTED: छूट के आधार पर निवल मासिक वेतन के अधीन समय के अधीन मासिक वेतन के आधार पर निवल मासिक वेतन शुल्क प्रभार
--------------------------------------------------------

In [6]:
def translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device):
    model.eval()

    src_tokens = tokenizer_src.encode(sentence).ids

    sos_id = tokenizer_src.token_to_id('[SOS]')
    eos_id = tokenizer_src.token_to_id('[EOS]')
    pad_id = tokenizer_src.token_to_id('[PAD]')

    src_tokens = [sos_id] + src_tokens + [eos_id]

    if len(src_tokens) < config['seq_len']:
        src_tokens += [pad_id] * (config['seq_len'] - len(src_tokens))
    else:
        src_tokens = src_tokens[:config['seq_len']]

    source = torch.tensor(src_tokens).unsqueeze(0).to(device)

    source_mask = (source != pad_id).unsqueeze(1).unsqueeze(2).int().to(device)

    output_tokens = greedy_decode(
        model, source, source_mask,
        tokenizer_src, tokenizer_tgt,
        config['seq_len'], device
    )
    translated = tokenizer_tgt.decode(output_tokens.detach().cpu().numpy())

    return translated

In [8]:
sentence = "Testing the trained model "

output = translate_sentence(
    sentence,
    model,
    tokenizer_src,
    tokenizer_tgt,
    config,
    device
)

print("Input:", sentence)
print("Output:", output)

Input: Testing the trained model 
Output: मॉडल प्रशिक्षण मॉडल का पुनः मॉडल
